## ML_1017: Implement BatchNorm

**Difficulty**: Medium | **TorchCode**: #07

### Problem
Implement Batch Normalization with train/eval modes. In training, use batch statistics and update running mean/variance with momentum. In inference, use running statistics only.

### Formula
$$\hat{x} = \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \varepsilon}}, \quad y = \gamma\hat{x} + \beta$$
$$\text{running\_mean} \leftarrow (1-m)\,\text{running\_mean} + m\,\mu_B$$

In [ ]:
import torch

def my_batch_norm(x, gamma, beta, running_mean, running_var,
                  eps=1e-5, momentum=0.1, training=True):
    if training:
        batch_mean = x.mean(dim=0)
        batch_var = x.var(dim=0, unbiased=False)
        running_mean.mul_(1 - momentum).add_(momentum * batch_mean.detach())
        running_var.mul_(1 - momentum).add_(momentum * batch_var.detach())
        mean, var = batch_mean, batch_var
    else:
        mean, var = running_mean, running_var
    x_norm = (x - mean) / torch.sqrt(var + eps)
    return gamma * x_norm + beta

In [ ]:
import torch

# --- Test 1: Training mode — zero mean per feature ---
x = torch.randn(8, 4)
gamma, beta = torch.ones(4), torch.zeros(4)
running_mean, running_var = torch.zeros(4), torch.ones(4)
out = my_batch_norm(x, gamma, beta, running_mean, running_var, training=True)
assert out.shape == x.shape
assert torch.allclose(out.mean(dim=0), torch.zeros(4), atol=1e-5)

# --- Test 2: Training — numerical correctness and running stats update ---
torch.manual_seed(0)
x = torch.randn(16, 8)
gamma, beta = torch.randn(8), torch.randn(8)
running_mean, running_var = torch.zeros(8), torch.ones(8)
momentum = 0.1
out = my_batch_norm(x, gamma, beta, running_mean, running_var, momentum=momentum, training=True)
mean_b = x.mean(dim=0); var_b = x.var(dim=0, unbiased=False)
ref = gamma * (x - mean_b) / torch.sqrt(var_b + 1e-5) + beta
assert torch.allclose(out, ref, atol=1e-4)
assert torch.allclose(running_mean, (1 - momentum) * torch.zeros(8) + momentum * mean_b, atol=1e-6)

# --- Test 3: Inference — uses running statistics ---
torch.manual_seed(0)
x = torch.randn(4, 8)
gamma, beta = torch.randn(8), torch.randn(8)
rm, rv = torch.randn(8), torch.rand(8) + 0.5
out = my_batch_norm(x, gamma, beta, rm.clone(), rv.clone(), training=False)
ref = gamma * (x - rm) / torch.sqrt(rv + 1e-5) + beta
assert torch.allclose(out, ref, atol=1e-4)

# --- Test 4: Gradient flow ---
x = torch.randn(4, 8, requires_grad=True)
gamma = torch.ones(8, requires_grad=True)
beta = torch.zeros(8, requires_grad=True)
my_batch_norm(x, gamma, beta, torch.zeros(8), torch.ones(8), training=True).sum().backward()
assert x.grad is not None and gamma.grad is not None and beta.grad is not None

print('All tests passed!')